# 03 - PHASE 0 GATE: decomposition of the efficiency gap on REAL logits

AGENTS.md Sec 5. The evidence that motivated this project came from synthetic
inject-then-recover, which is circular. Before anything else the decomposition must
replicate on REAL logits.

Measure how much of the efficiency gap is closed, SEPARATELY, by:
1. a single global **temperature** (1 free parameter),
2. a per-sample offset indexed by **free energy** `E(x) = -logsumexp(logits)` (n_bins params),
3. a per-**class** offset fit on abundant data (K params).

**PRE-REGISTERED PASS CRITERION (Sec 5):** (3) must close a gap SUBSTANTIALLY larger than
(1) and (2), with **non-overlapping CIs**. If it does not, the hypothesis that the structure
lives at the class level does not hold on real data -> **STOP, do not enter Phase 1.**

## AMENDMENT 1 (2026-08-03) - read reports/protocol_amendments.md

The first run of this notebook returned FAIL with **every** component negative
(temperature +0.014, all energy bins negative, per-class offset -5.615). A result where every
method makes sets BIGGER signals a broken measurement, not a finding.

Cause: the split-conformal quantile uses level `ceil((n+1)(1-alpha))/n`, which **depends on n**.
A 50-sample class group therefore targets that class's ~98th percentile while the pooled global
group targets the ~95th. That level mismatch - not class structure - produced the negative gaps,
and it explains the whole pattern (energy got worse as bins grew, i.e. as samples/bin shrank).

Verified on synthetic at realistic accuracy: per-class gap **-5.98 (conformal) -> +1.98
(empirical) vs +1.84 (abundant-data oracle)** - so it was the level, not finite-sample noise.

The structure measurement now uses **level-matched empirical quantiles (PRIMARY)**. Both
estimators are computed and reported; `conformal` is what DEPLOYMENT uses and its coverage
validity is guarded separately by tests/test_coverage_validity.py (Sec 8.7).

**CIFAR-100 caveat:** this is the pipeline-DEBUG dataset (100 classes, 100 test images/class,
alpha=0.01 infeasible - see reports/phase0_checkpoint_gate.md). A CIFAR-100 result does NOT
decide the Phase-0 gate; Pl@ntNet does. Here we verify the code and read the direction.


## 1. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'
DATASET    = 'cifar100'
BACKBONE   = 'resnet50_self'

ALPHAS     = (0.05, 0.1)       # alpha=0.01 infeasible at 100 img/class (pre-registered)
ESTIMATORS = ('empirical', 'conformal')   # 'empirical' = PRIMARY (Amendment 1)
N_SPLITS   = 100              # >=100 random cal/eval splits (Sec 8.4)
BIN_GRID   = (2, 5, 10, 20, 50, 100)
SEED = 42
EMB_DIR = f'{DRIVE_ROOT}/embeddings/{DATASET}/{BACKBONE}'
# =======================================================================
print('EMB_DIR =', EMB_DIR, '| alphas', ALPHAS, '| estimators', ESTIMATORS)


## 2. Mount Drive + repo + env + seed


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Load TEST-set logits (the conformal cal/eval pool)


In [ ]:
import numpy as np, os
d = np.load(os.path.join(EMB_DIR,'test.npz'))
logits, labels = d['logits'], d['labels']
n, K = logits.shape
print(f'logits={logits.shape} classes={K} samples/class~{n//K}')
print('accuracy:', round(float((logits.argmax(1)==labels).mean()),4))


## 4. Decomposition over >=100 random cal/eval splits, for BOTH estimators (Sec 8.4)


In [ ]:
from pcc.eval import decomposition as dc
from pcc.eval.stats import mean_ci
import numpy as np

S = 1 - dc.temperature_softmax(logits, 1.0)
results = {}
for estimator in ESTIMATORS:
    for alpha in ALPHAS:
        acc = {'temperature': [], 'class_offset': [],
               **{f'energy_b{b}': [] for b in BIN_GRID}}
        rng = np.random.default_rng(SEED)
        for _ in range(N_SPLITS):
            idx = rng.permutation(n); cal, ev = idx[:n//2], idx[n//2:]
            acc['temperature'].append(dc.gap_from_global_temperature(
                logits, labels, alpha, cal, ev, estimator=estimator)['gap_closed'])
            acc['class_offset'].append(dc.gap_from_per_class_offset(
                S, labels, K, alpha, cal, ev, estimator=estimator)['gap_closed'])
            sw = dc.phase0_energy_bin_sweep(logits, S, labels, alpha, cal, ev,
                                            bin_grid=BIN_GRID, estimator=estimator)
            for b in BIN_GRID:
                acc[f'energy_b{b}'].append(sw[b]['gap_closed'])
        results[(estimator, alpha)] = {k: mean_ci(v) for k, v in acc.items()}
        print(f'--- estimator={estimator}  alpha={alpha} ---')
        for k, v in results[(estimator, alpha)].items():
            print(f"  {k:16s} gap={v['mean']:+7.3f}  95% CI [{v['ci_low']:+.3f}, {v['ci_high']:+.3f}]")


## 5. Gate verdict - non-overlapping CIs (Sec 5). PRIMARY = `empirical` (Amendment 1)


In [ ]:
verdicts = {}
for (estimator, alpha) in results:
    r = results[(estimator, alpha)]
    cls = r['class_offset']
    rivals = {k: v for k, v in r.items() if k != 'class_offset'}
    best_name = max(rivals, key=lambda k: rivals[k]['mean'])
    best = rivals[best_name]
    non_overlap = cls['ci_low'] > best['ci_high']
    key = f'{estimator}|{alpha}'
    verdicts[key] = {'class_gap': cls['mean'], 'best_rival': best_name,
                     'rival_gap': best['mean'], 'non_overlapping_CI': bool(non_overlap),
                     'pass': bool(non_overlap and cls['mean'] > best['mean'])}
    print(f"[{estimator:9s}] alpha={alpha}: class={cls['mean']:+.3f} "
          f"[{cls['ci_low']:+.3f},{cls['ci_high']:+.3f}] vs best rival {best_name}="
          f"{best['mean']:+.3f} [{best['ci_low']:+.3f},{best['ci_high']:+.3f}] "
          f"-> {'PASS' if verdicts[key]['pass'] else 'FAIL'}")

# PRIMARY verdict uses ONLY the level-matched estimator (Amendment 1)
primary = {k: v for k, v in verdicts.items() if k.startswith('empirical')}
overall = 'PASS' if all(v['pass'] for v in primary.values()) else 'FAIL'
print('\nPRIMARY (empirical / level-matched) CIFAR-100 DEBUG direction:', overall)
print('NOTE: this does NOT decide the Phase-0 gate - Pl@ntNet does (see cell 0).')


## 6. Write report


In [ ]:
import time
from pcc.utils.io import write_report
clean = {f'{est}|{a}': {k: {kk: float(vv) for kk, vv in v.items()}
                        for k, v in r.items()} for (est, a), r in results.items()}
report = write_report('pcc/reports', f'03_phase0_decomposition_{DATASET}',
    hypothesis='a per-CLASS offset closes a substantially larger efficiency gap than a global '
               'temperature or a per-sample energy-indexed offset, on REAL logits',
    pass_criteria='class_offset gap > best rival AND non-overlapping 95% CIs at every alpha, '
                  'judged on the LEVEL-MATCHED (empirical) estimator per Amendment 1; energy '
                  'swept over bin counts so the win is not a parameter-count artefact. '
                  'CIFAR-100 is DEBUG ONLY and does not decide the gate.',
    config=dict(dataset=DATASET, backbone=BACKBONE, alphas=list(ALPHAS),
                n_splits=N_SPLITS, bin_grid=list(BIN_GRID), estimators=list(ESTIMATORS),
                primary_estimator='empirical',
                amendment='reports/protocol_amendments.md#amendment-1'),
    seed=SEED, results={'by_estimator_alpha': clean, 'verdicts': verdicts,
                        'primary_estimator': 'empirical', 'debug_only': True},
    conclusion=f'{overall} (CIFAR-100 debug direction, empirical estimator; not the gate verdict)',
    started_at=time.time())
print('report:', report)
